<a href="https://colab.research.google.com/github/zakariazemmahi/waste-detection-yolov8/blob/main/Models/Application_de_comptur_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **⚙️ Étape 1 – Installer les dépendances**

In [ ]:
!pip install streamlit ultralytics
!npm install -g localtunnel


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

In [ ]:
# Avant votre code Streamlit, ajoutez ce bloc d'installation
import subprocess
import sys

def install_spacy_models():
    try:
        # Essayer d'abord d'installer le modèle transformer (meilleure performance)
        subprocess.run([sys.executable, "-m", "spacy", "download", "fr_dep_news_trf"], check=True)
    except:
        try:
            # Fallback sur le modèle moyen si échec
            subprocess.run([sys.executable, "-m", "spacy", "download", "fr_core_news_md"], check=True)
        except Exception as e:
            st.error(f"Échec de l'installation des modèles SpaCy : {str(e)}")
            st.stop()

install_spacy_models()

# **📝 Étape 2 – Créer ton fichier app.py Streamlit**

In [ ]:
%%writefile app.py
import streamlit as st
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import requests
import streamlit.components.v1 as components
import cv2
from io import BytesIO
import base64
import plotly.express as px
import plotly.graph_objects as go
import gc
import time
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import re
import json
from pathlib import Path


# Configuration de la page
st.set_page_config(
    page_title="SmartWasteDetection",
    page_icon="♻",
    layout="wide",
    initial_sidebar_state="expanded"
)

# CSS personnalisé moderne (identique à votre version originale)
# ... [Votre CSS existant reste inchangé] ...
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap');

    * {
        font-family: 'Poppins', sans-serif;
    }

    .main-header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 3rem 2rem;
        border-radius: 20px;
        text-align: center;
        margin-bottom: 2rem;
        box-shadow: 0 20px 40px rgba(0, 0, 0, 0.1);
        position: relative;
        overflow: hidden;
    }

    .main-header::before {
        content: '';
        position: absolute;
        top: -50%;
        left: -50%;
        width: 200%;
        height: 200%;
        background: linear-gradient(45deg, transparent, rgba(255,255,255,0.1), transparent);
        animation: shine 3s infinite;
    }

    @keyframes shine {
        0% { transform: translateX(-100%) translateY(-100%) rotate(45deg); }
        100% { transform: translateX(100%) translateY(100%) rotate(45deg); }
    }

    .main-header h1 {
        color: white;
        font-size: 3.5rem;
        margin: 0;
        font-weight: 700;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
        position: relative;
        z-index: 1;
    }

    .main-header p {
        color: rgba(255,255,255,0.9);
        font-size: 1.3rem;
        margin-top: 1rem;
        position: relative;
        z-index: 1;
        font-weight: 300;
    }

    .detection-card {
        background: linear-gradient(145deg, #ffffff, #f0f2f6);
        padding: 2rem;
        border-radius: 20px;
        box-shadow: 0 10px 30px rgba(0, 0, 0, 0.1);
        border: 1px solid rgba(255,255,255,0.2);
        margin: 1rem 0;
        transition: transform 0.3s ease, box-shadow 0.3s ease;
    }

    .detection-card:hover {
        transform: translateY(-5px);
        box-shadow: 0 20px 40px rgba(0, 0, 0, 0.15);
    }

    .stat-card {
        background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
        padding: 2rem;
        border-radius: 15px;
        text-align: center;
        color: white;
        box-shadow: 0 10px 25px rgba(79, 172, 254, 0.3);
        transition: transform 0.3s ease;
    }

    .stat-card:hover {
        transform: scale(1.05);
    }

    .stat-number {
        font-size: 2.5rem;
        font-weight: 700;
        margin: 0;
    }

    .stat-label {
        font-size: 0.9rem;
        opacity: 0.9;
        font-weight: 300;
    }

    .upload-zone {
        border: 3px dashed #667eea;
        border-radius: 20px;
        padding: 3rem;
        text-align: center;
        background: linear-gradient(135deg, rgba(102, 126, 234, 0.1), rgba(118, 75, 162, 0.1));
        margin: 2rem 0;
        transition: all 0.3s ease;
    }

    .upload-zone:hover {
        border-color: #764ba2;
        background: linear-gradient(135deg, rgba(102, 126, 234, 0.2), rgba(118, 75, 162, 0.2));
    }

    .sidebar-info {
        background: linear-gradient(135deg, #ff9a56 0%, #ff6b95 100%);
        padding: 1.5rem;
        border-radius: 15px;
        color: white;
        text-align: center;
        margin: 1rem 0;
    }

    .progress-container {
        background: rgba(255,255,255,0.1);
        border-radius: 10px;
        padding: 1rem;
        margin: 1rem 0;
    }

    .image-container {
        border-radius: 15px;
        overflow: hidden;
        box-shadow: 0 10px 30px rgba(0, 0, 0, 0.2);
        margin: 1rem 0;
        transition: transform 0.3s ease;
    }

    .image-container:hover {
        transform: scale(1.02);
    }

    .results-section {
        background: linear-gradient(135deg, #ffecd2 0%, #fcb69f 100%);
        padding: 2rem;
        border-radius: 20px;
        margin: 2rem 0;
    }

    .waste-type-badge {
        display: inline-block;
        padding: 0.5rem 1rem;
        background: linear-gradient(135deg, #ff6b6b, #ee5a24);
        color: white;
        border-radius: 20px;
        margin: 0.25rem;
        font-weight: 600;
        box-shadow: 0 5px 15px rgba(255, 107, 107, 0.3);
    }

    .non-waste-badge {
        display: inline-block;
        padding: 0.5rem 1rem;
        background: linear-gradient(135deg, #00d2d3, #54a0ff);
        color: white;
        border-radius: 20px;
        margin: 0.25rem;
        font-weight: 600;
        box-shadow: 0 5px 15px rgba(84, 160, 255, 0.3);
    }

    .stButton > button {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        border: none;
        padding: 0.75rem 2rem;
        border-radius: 25px;
        font-weight: 600;
        transition: all 0.3s ease;
        box-shadow: 0 5px 15px rgba(102, 126, 234, 0.3);
    }

    .stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 10px 25px rgba(102, 126, 234, 0.4);
    }

    .analysis-header {
        text-align: center;
        color: #333;
        font-size: 2rem;
        font-weight: 600;
        margin: 2rem 0;
        background: linear-gradient(135deg, #667eea, #764ba2);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
    }
</style>
""", unsafe_allow_html=True)
# Configuration de la base de données
def init_database():
    """Initialise la base de données SQLite"""
    conn = sqlite3.connect('waste_detection.db')
    cursor = conn.cursor()

    # Table pour les statistiques des déchets par lieu
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS waste_statistics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            location TEXT NOT NULL,
            waste_type TEXT NOT NULL,
            count INTEGER DEFAULT 1,
            date_added TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    # Table pour l'historique des détections
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS detection_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            location TEXT NOT NULL,
            waste_type TEXT NOT NULL,
            confidence REAL,
            image_name TEXT,
            detection_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    # Index pour améliorer les performances
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_location ON waste_statistics(location)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_waste_type ON waste_statistics(waste_type)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_date ON detection_history(detection_date)')

    conn.commit()
    conn.close()

def get_locations_list():
    """Retourne la liste des lieux disponibles"""
    return [
        "TD1",
        "Amphi 250",
        "Amphi 450",
        "Département Math_Info",
        "Département Mécanique structure",
        "Département Énergétique",
        "Département Civil",
        "Département procédé ",
        "Département AEEE",
        "Buvette",
        "Bibliothèque",
        "TD2",
        "Autre"
    ]


def add_waste_detection(location, waste_type, confidence, image_name):
    """Ajoute une détection de déchet à la base de données"""
    conn = sqlite3.connect('waste_detection.db')
    cursor = conn.cursor()

    try:
        # Ajouter à l'historique
        cursor.execute('''
            INSERT INTO detection_history (location, waste_type, confidence, image_name)
            VALUES (?, ?, ?, ?)
        ''', (location, waste_type, confidence, image_name))

        # Vérifier si cette combinaison lieu/type existe déjà
        cursor.execute('''
            SELECT id, count FROM waste_statistics
            WHERE location = ? AND waste_type = ?
        ''', (location, waste_type))

        result = cursor.fetchone()

        if result:
            # Mettre à jour le compteur existant
            cursor.execute('''
                UPDATE waste_statistics
                SET count = count + 1, last_updated = CURRENT_TIMESTAMP
                WHERE id = ?
            ''', (result[0],))
        else:
            # Créer une nouvelle entrée
            cursor.execute('''
                INSERT INTO waste_statistics (location, waste_type, count)
                VALUES (?, ?, 1)
            ''', (location, waste_type))

        conn.commit()
        return True

    except Exception as e:
        st.error(f"Erreur lors de l'ajout à la base de données: {str(e)}")
        return False
    finally:
        conn.close()

def get_statistics_by_location():
    """Récupère les statistiques par lieu"""
    conn = sqlite3.connect('waste_detection.db')

    try:
        df = pd.read_sql_query('''
            SELECT location, waste_type, count, last_updated
            FROM waste_statistics
            ORDER BY location, count DESC
        ''', conn)
        return df
    except Exception as e:
        st.error(f"Erreur lors de la récupération des statistiques: {str(e)}")
        return pd.DataFrame()
    finally:
        conn.close()

def get_detection_history(days=7):
    """Récupère l'historique des détections"""
    conn = sqlite3.connect('waste_detection.db')

    try:
        df = pd.read_sql_query('''
            SELECT location, waste_type, confidence, image_name, detection_date
            FROM detection_history
            WHERE detection_date >= datetime('now', '-{} days')
            ORDER BY detection_date DESC
        '''.format(days), conn)
        return df
    except Exception as e:
        st.error(f"Erreur lors de la récupération de l'historique: {str(e)}")
        return pd.DataFrame()
    finally:
        conn.close()

def get_total_statistics():
    """Récupère les statistiques globales avec gestion des erreurs améliorée"""
    conn = sqlite3.connect('waste_detection.db')
    cursor = conn.cursor()

    try:
        # Initialiser le dictionnaire de résultats avec des valeurs par défaut
        stats = {
            'waste_by_type': [],
            'waste_by_location': [],
            'recent_detections': 0,
            'top_location': ("Aucun lieu", 0)
        }

        # Total des déchets par type
        cursor.execute('''
            SELECT waste_type, SUM(count) as total
            FROM waste_statistics
            GROUP BY waste_type
            ORDER BY total DESC
        ''')
        waste_by_type = cursor.fetchall()
        if waste_by_type:  # Vérifier si la liste n'est pas vide
            stats['waste_by_type'] = waste_by_type

        # Total des déchets par lieu
        cursor.execute('''
            SELECT location, SUM(count) as total
            FROM waste_statistics
            GROUP BY location
            ORDER BY total DESC
        ''')
        waste_by_location = cursor.fetchall()
        if waste_by_location:
            stats['waste_by_location'] = waste_by_location

        # Statistiques récentes (7 derniers jours)
        cursor.execute('''
            SELECT COUNT(*) as recent_detections
            FROM detection_history
            WHERE detection_date >= datetime('now', '-7 days')
        ''')
        recent_count = cursor.fetchone()
        if recent_count and recent_count[0] is not None:
            stats['recent_detections'] = recent_count[0]

        # Lieu le plus problématique
        cursor.execute('''
            SELECT location, SUM(count) as total
            FROM waste_statistics
            GROUP BY location
            ORDER BY total DESC
            LIMIT 1
        ''')
        top_location = cursor.fetchone()
        if top_location and len(top_location) == 2:
            stats['top_location'] = top_location

        return stats

    except sqlite3.Error as e:
        st.error(f"Erreur SQL lors du calcul des statistiques: {str(e)}")
        return {
            'waste_by_type': [],
            'waste_by_location': [],
            'recent_detections': 0,
            'top_location': ("Erreur", 0)
        }
    except Exception as e:
        st.error(f"Erreur inattendue lors du calcul des statistiques: {str(e)}")
        return {
            'waste_by_type': [],
            'waste_by_location': [],
            'recent_detections': 0,
            'top_location': ("Erreur", 0)
        }
    finally:
        conn.close()

def display_statistics_dashboard():
    """Affiche le tableau de bord des statistiques"""
    st.markdown("## 📊 Tableau de Bord des Statistiques")

    # Initialiser la base de données
    init_database()

    # Récupérer les données
    stats_df = get_statistics_by_location()
    total_stats = get_total_statistics()
    history_df = get_detection_history()

    if stats_df.empty:
        st.info("🔍 Aucune donnée disponible. Commencez par analyser des images avec localisation.")
        return

    # Métriques principales
    st.markdown("### 📈 Métriques Globales")
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        total_waste = stats_df['count'].sum()
        st.metric("🗑 Total Déchets", total_waste)

    with col2:
        unique_locations = stats_df['location'].nunique()
        st.metric("📍 Lieux Surveillés", unique_locations)

    with col3:
        unique_types = stats_df['waste_type'].nunique()
        st.metric("♻ Types Différents", unique_types)

    with col4:
        recent_detections = total_stats.get('recent_detections', 0)
        st.metric("📅 Cette Semaine", recent_detections)

    # Graphiques
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### 🏢 Déchets par Lieu")
        location_stats = stats_df.groupby('location')['count'].sum().reset_index()
        fig_location = px.bar(
            location_stats,
            x='location',
            y='count',
            title="Distribution des déchets par lieu",
            color='count',
            color_continuous_scale='Reds'
        ).update_layout(xaxis_tickangle=45)  # Corrected line
        st.plotly_chart(fig_location, use_container_width=True)

    with col2:
        st.markdown("### ♻ Types de Déchets")
        type_stats = stats_df.groupby('waste_type')['count'].sum().reset_index()
        fig_type = px.pie(
            type_stats,
            values='count',
            names='waste_type',
            title="Répartition par type de déchet"
        )
        st.plotly_chart(fig_type, use_container_width=True)

    # Rest of your function remains the same...

# Fonction modifiée pour l'interface de détection avec sélection de lieu
def enhanced_detection_interface():
    """Interface de détection améliorée avec sélection de lieu"""

    # Initialiser la base de données
    init_database()

    # Section upload avec sélection de lieu
    st.markdown("### 📍 Localisation et Upload")

    col1, col2 = st.columns([1, 2])

    with col1:
        selected_location = st.selectbox(
            "🏢 Sélectionnez le lieu de détection:",
            options=get_locations_list(),
            help="Choisissez où l'image a été prise pour enrichir les statistiques"
        )

        st.info(f"📍 Lieu sélectionné: *{selected_location}*")

    with col2:
        uploaded_files = st.file_uploader(
            "📤 Glissez-déposez vos images ici",
            type=["jpg", "jpeg", "png"],
            accept_multiple_files=True,
            help="Formats supportés: JPG, JPEG, PNG"
        )

    return selected_location, uploaded_files

# Fonction pour sauvegarder les détections
def save_detections_to_db(waste_objects, classifications, selected_location, image_name):
    """Sauvegarde les détections dans la base de données"""
    saved_count = 0

    for i, waste_obj in enumerate(waste_objects):
        if i < len(classifications) and classifications[i][0]:
            waste_type = classifications[i][0]
            confidence = classifications[i][1]

            if add_waste_detection(selected_location, waste_type, confidence, image_name):
                saved_count += 1

    return saved_count

def show_save_confirmation(waste_objects, classifications, selected_location, image_name):
    """Affiche une interface de confirmation pour sauvegarder les détections"""

    if not waste_objects:
        return

    st.markdown("### 💾 Sauvegarde des Détections")

    # Résumé des détections à sauvegarder
    st.write(f"*Lieu:* {selected_location}")
    st.write(f"*Image:* {image_name}")

    detected_types = []
    for i, waste_obj in enumerate(waste_objects):
        if i < len(classifications) and classifications[i][0]:
            detected_types.append(classifications[i][0])

    if detected_types:
        st.write(f"*Déchets détectés:* {', '.join(detected_types)}")

        col1, col2 = st.columns(2)

        with col1:
            if st.button("💾 Sauvegarder dans la Base",
                        type="primary",
                        key=f"save_button_{image_name}"):  # Add unique key here
                saved_count = save_detections_to_db(waste_objects, classifications, selected_location, image_name)
                if saved_count > 0:
                    st.success(f"✅ {saved_count} détection(s) sauvegardée(s)!")
                    st.balloons()
                else:
                    st.error("❌ Erreur lors de la sauvegarde")

        with col2:
            if st.button("🔍 Voir les Statistiques",
                         key=f"stats_button_{image_name}"):  # Add unique key here
                st.session_state['show_stats'] = True
    else:
        st.info("ℹ Aucun déchet classifié à sauvegarder")

# CSS pour améliorer l'apparence des statistiques
def add_statistics_css():
    """Ajoute du CSS spécifique pour les statistiques"""
    st.markdown("""
    <style>
    .metric-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.5rem;
        border-radius: 15px;
        color: white;
        text-align: center;
        box-shadow: 0 10px 25px rgba(102, 126, 234, 0.3);
        margin: 0.5rem 0;
    }

    .location-header {
        background: linear-gradient(135deg, #ff9a56 0%, #ff6b95 100%);
        padding: 1rem;
        border-radius: 10px;
        color: white;
        font-weight: 600;
        margin: 1rem 0 0.5rem 0;
    }

    .stats-container {
        background: rgba(255, 255, 255, 0.05);
        padding: 2rem;
        border-radius: 20px;
        backdrop-filter: blur(10px);
        border: 1px solid rgba(255, 255, 255, 0.1);
    }
    </style>
    """, unsafe_allow_html=True)


# Chargement des modèles YOLO (identique à votre version originale)
@st.cache_resource
def load_models():
    """Charge les modèles YOLO avec gestion d'erreur améliorée"""
    try:
        with st.spinner("🤖 Chargement du modèle de détection..."):
            model_detect = YOLO("/content/drive/MyDrive/yolov8_best_smartdetection.pt")

        with st.spinner("🔍 Chargement du modèle de classification..."):
            model_classify = YOLO("/content/drive/MyDrive/yolov8_best.pt")

        return model_detect, model_classify
    except Exception as e:
        st.error(f"❌ Erreur lors du chargement des modèles : {str(e)}")
        return None, None

# Chargement du modèle NLP français
try:
    nlp = spacy.load("fr_dep_news_trf")  # Modèle transformer plus précis
except:
    nlp = spacy.load("fr_core_news_md")  # Fallback
    st.sidebar.warning("Modèle NLP de base utilisé (pour une meilleure performance, installez fr_dep_news_trf)")

# Chargement des données FAQ et localisations
def load_data():
    data_dir = Path("data")
    faq_path = data_dir / "faq.json"
    locations_path = data_dir / "locations.json"

    data_dir.mkdir(exist_ok=True)

    # Données par défaut si fichiers inexistants
    default_faq = {
        "Puis-je recycler une bouteille en plastique ?": "✅ Oui, les bouteilles en plastique vont dans le bac de recyclage.",
    "Puis-je recycler une bouteille en verre ?": "✅ Oui, le verre est 100% recyclable.",
    "Où jeter les piles usagées ?": "❌ Ne jamais jeter les piles à la poubelle. Apportez-les dans une borne spéciale.",
    "Que faire avec les déchets organiques ?": "♻ Ils peuvent être compostés ou déposés dans un bac dédié.",
    "Les canettes métalliques sont-elles recyclables ?": "✅ Oui, elles vont dans le bac de tri.",
    "Puis-je jeter un masque chirurgical dans le recyclage ?": "❌ Non, les masques vont à la poubelle classique.",
    "Que faire des bouchons en plastique ?": "✅ Ils sont recyclables, mais vous pouvez aussi les donner à des associations.",
    "Puis-je jeter les cartons de pizza dans le bac de tri ?": "❌ Non s’ils sont sales ou gras. ✅ Oui s’ils sont propres.",
    "Les sacs plastiques sont-ils recyclables ?": "❌ Non, ils doivent aller dans la poubelle ordinaire ou être rapportés en magasin.",
    "Puis-je jeter un téléphone cassé dans la poubelle ?": "❌ Non, il faut le déposer dans une déchèterie ou point de collecte.",
    "Que faire avec les médicaments périmés ?": "💊 Apportez-les en pharmacie. Ne jamais les jeter dans la poubelle ou les toilettes.",
    "Où jeter les ampoules usagées ?": "💡 Apportez-les dans les points de collecte spécifiques (ex : magasins, déchèterie).",
    "Puis-je recycler un pot de yaourt ?": "✅ Oui, s'il est propre et vide.",
    "Puis-je jeter un vêtement usé dans le bac de tri ?": "❌ Non, utilisez les bornes textiles ou donnez-les à une association.",
    "Où jeter une boîte de conserve ?": "✅ Dans le bac de recyclage, après l’avoir vidée.",
    "Que faire avec les déchets électroniques ?": "🖥 Rapportez-les en magasin ou en déchèterie.",
    "Les cartons sont-ils toujours recyclables ?": "✅ Oui, sauf s’ils sont souillés (gras, alimentaires).",
    "Peut-on recycler les mouchoirs usagés ?": "❌ Non, jetez-les dans la poubelle ordinaire.",
    "Puis-je recycler les barquettes en aluminium ?": "✅ Oui, si elles sont propres.",
    "Les emballages plastiques sont-ils tous recyclables ?": "❌ Non, seuls certains types (bouteilles, flacons) le sont. Vérifiez les consignes locales.",
    "Puis-je recycler un stylo usé ?": "❌ Non, il va à la poubelle classique.",
    "Puis-je jeter les restes de nourriture dans le compost ?": "♻ Oui pour les épluchures, ❌ non pour la viande ou les produits laitiers.",
    "Est-ce que le papier imprimé est recyclable ?": "✅ Oui, sauf s’il est plastifié ou trop sali.",
    "Où jeter un CD ou DVD ?": "❌ À la poubelle classique ou dans une déchèterie.",
    "Puis-je recycler une brosse à dents ?": "❌ Non, sauf modèles spécifiques en plastique recyclable.",
    "Puis-je jeter une casserole abîmée dans le tri ?": "❌ Non, apportez-la en déchèterie.",
    "Les journaux et magazines sont-ils recyclables ?": "✅ Oui, sans plastique ou reliures métalliques.",
    "Les capsules de café sont-elles recyclables ?": "☕ En général non, sauf programmes spécifiques (ex : Nespresso).",
    "Les cartons de lait sont-ils recyclables ?": "✅ Oui, dans la plupart des communes. Vérifiez localement.",
    "Les bouchons en liège sont-ils recyclables ?": "✅ Oui, dans certaines filières ou pour des associations de recyclage.",
    }

    default_locations = {
        "cafétéria": {
        "poubelle": "📍 À gauche de la sortie de la cafétéria, à côté de je du football par table.",
        "entrée": "🚪 L’entrée principale est a coté du departemant mecanique."
    },
    "hall principal": {
        "poubelle": "📍 À droite du comptoir d’accueil, sous l’escalier.",
        "entrée": "🚪 Entrée par la porte A, côté parking."
    },
    "amphi A": {
        "poubelle": "📍 Derrière la porte principale.",
        "entrée": "🚪 Entrée au fond du couloir vers buvette."
    },
    "td1": {
        "poubelle": "📍 Devant la salle TD1.",
        "entrée": "🚪 Entrée par le couloir , vers departemant math_info."
    },
    "amphi 250": {
        "poubelle": "📍 Derrière la porte principale.",
        "entrée": "🚪 Entrée au fond du couloir vers buvette"
    },
    "amphi 450": {
        "poubelle": "À gauche en entrant,de la couloire de departemant math info.",
        "entrée": "🚪 couloire vers buvette."
    },
    "département math_info": {
        "poubelle": "📍 Juste à l’entrée du département, à côté de la salle des professeurs.",
        "entrée": "🚪 Entrée par le couloir vers les terrains du sports."
    },
    "mécanique": {
        "poubelle": "📍 En face du laboratoire de matériaux.",
        "entrée": "🚪 Entrée côté atelier."
    },
    "énergétique": {
        "poubelle": "📍 À côté du banc d’essai thermique.",
        "entrée": "🚪 Entrée bâtiment E, porte 3."
    },
    "buvette": {
        "poubelle": "📍 À l'entréé du buvette.",
        "entrée": "🚪 Entrée face au Departemant énergitique."
    },
    "bibliothèque": {
        "poubelle": "📍 À l’entrée de la bibloitique et a coté des escalier.",
        "entrée": "🚪 Entrée côté sud, porte BIB."
    },
    "td2": {
        "poubelle": "📍 à l'entrée du td.",
        "entrée": "🚪 è l'entréé de l'école."
    }
  }

    try:
        with open(faq_path, 'r', encoding='utf-8') as f:
            faq = json.load(f)
    except:
        faq = default_faq
        with open(faq_path, 'w', encoding='utf-8') as f:
            json.dump(faq, f, ensure_ascii=False, indent=2)

    try:
        with open(locations_path, 'r', encoding='utf-8') as f:
            locations = json.load(f)
    except:
        locations = default_locations
        with open(locations_path, 'w', encoding='utf-8') as f:
            json.dump(locations, f, ensure_ascii=False, indent=2)

    return faq, locations

faq, locations = load_data()

# Initialisation des embeddings FAQ
questions = list(faq.keys())
faq_vectors = np.array([nlp(q).vector for q in questions])

# Système de synonymes et normalisation
synonyms = {
    "td 1": "td1",
    "td 2": "td2",
    "amphi a": "amphi a",
    "math info": "département math_info",
    "méca": "mécanique",
    "energie": "énergétique",
    "td 1": "TD1",
    "amphi a": "amphi a",
    "amphi 250": "amphi 250",
    "amphi 450": "amphi 450",
    "département math info": "département math_info",
    "math info": "département math_info",
    "département mécanique": "mécanique",
    "département énergétique": "énergétique"
}

def normalize_text(text):
    text = text.lower().strip()
    for syn, main in synonyms.items():
        text = text.replace(syn, main)
    text = re.sub(r'[^\w\s]', '', text)
    return text

# Classe du Chatbot amélioré
class WasteChatbot:
    def _init_(self):
        self.faq = faq
        self.locations = locations
        self.vector_cache = {}
        self.embedding_dim = 300
        self.empty_embedding = np.zeros(self.embedding_dim)

        # Initialisation des statistiques
        self.stats = {
            'location_queries': 0,
            'faq_queries': 0,
            'unknown_queries': 0,
            'errors': 0
        }

    def get_valid_embedding(self, text):
        """Garantit toujours un embedding valide"""
        try:
            text = str(text).strip()

            if text in self.vector_cache:
                return self.vector_cache[text]

            if len(text) < 2:
                return self.empty_embedding

            doc = nlp(text)
            if doc.vector_norm > 0:
                embedding = doc.vector
            else:
                embedding = np.random.normal(0, 0.1, self.embedding_dim)

            self.vector_cache[text] = embedding
            return embedding

        except Exception as e:
            self.stats['errors'] += 1
            print(f"Erreur get_valid_embedding: {str(e)}")
            return self.empty_embedding

    def handle_location_query(self, query):
        """Version robuste de la recherche de localisation"""
        try:
            query = normalize_text(query)

            # Recherche exacte d'abord
            for location, info in self.locations.items():
                if location.lower() in query:
                    return info

            # Recherche sémantique
            query_embed = self.get_valid_embedding(query)
            best_score = 0
            best_match = None

            for location in self.locations:
                loc_embed = self.get_valid_embedding(location)
                score = cosine_similarity(
                    [query_embed],
                    [loc_embed]
                )[0][0]

                if score > best_score:
                    best_score = score
                    best_match = location

            return self.locations[best_match] if best_score > 0.6 else None

        except Exception as e:
            self.stats['errors'] += 1
            print(f"Erreur handle_location_query: {str(e)}")
            return None

    def process_query(self, query):
        """Version robuste du traitement des requêtes"""
        try:
            # Vérifier les lieux
            location_info = self.handle_location_query(query)
            if location_info:
                self.stats['location_queries'] += 1
                return f"{location_info['poubelle']}\nℹ {location_info['entrée']}"

            # Recherche dans la FAQ
            query_embed = self.get_valid_embedding(query)
            faq_embeddings = np.array([self.get_valid_embedding(q) for q in self.faq.keys()])

            similarities = cosine_similarity([query_embed], faq_embeddings)[0]
            best_idx = np.argmax(similarities)

            if similarities[best_idx] > 0.65:
                self.stats['faq_queries'] += 1
                return list(self.faq.values())[best_idx]
            else:
                self.stats['unknown_queries'] += 1
                return "Je n'ai pas trouvé d'information précise. Pouvez-vous reformuler ?"

        except Exception as e:
            self.stats['errors'] += 1
            print(f"Erreur process_query: {str(e)}")
            return "Désolé, un problème technique est survenu. Veuillez réessayer."

# Initialisation du chatbot
@st.cache_resource
def load_chatbot():
    return WasteChatbot()

# ... [Vos fonctions existantes process_image_detection, classify_waste, etc.] ...
def process_image_detection(image, model_detect):
    """Processus de détection des objets (déchets vs non-déchets)"""
    img_array = np.array(image)
    results = model_detect.predict(img_array, conf=0.25, verbose=False)

    waste_objects = []
    non_waste_objects = []

    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cls_id = int(box.cls)
            class_name = results[0].names[cls_id]
            conf = float(box.conf)
            bbox = box.xyxy.cpu().numpy().astype(int)[0]

            obj = {
                'class': class_name,
                'confidence': conf,
                'bbox': bbox
            }

            # Vérification si l'objet est un déchet
            if class_name.lower() == "dechet":
                waste_objects.append(obj)
            else:
                non_waste_objects.append(obj)

    return waste_objects, non_waste_objects, results[0] if results else None

def classify_waste(image, bbox, model_classify):
    """Classification du type de déchet pour les objets détectés comme déchets"""
    x1, y1, x2, y2 = bbox

    # Extraction de la région d'intérêt
    cropped = np.array(image)[y1:y2, x1:x2]

    if cropped.size == 0:
        return None, 0

    # Prédiction sur la région extraite
    results = model_classify.predict(cropped, conf=0.25, verbose=False)

    if results and results[0].boxes is not None and len(results[0].boxes) > 0:
        box = results[0].boxes[0]
        class_name = results[0].names[int(box.cls)]
        confidence = float(box.conf)
        return class_name, confidence

    return None, 0

def create_annotated_image(image, waste_objects, non_waste_objects, classifications):
    """Création d'une image annotée avec les détections"""
    # Créer une copie pour éviter de modifier l'original
    annotated_image = image.copy()
    draw = ImageDraw.Draw(annotated_image)

    # Charger une police
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 24)
        font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18)
    except:
        font = ImageFont.load_default()
        font_small = ImageFont.load_default()

    # Annoter les déchets (rouge) avec leur classification
    for i, obj in enumerate(waste_objects):
        x1, y1, x2, y2 = obj['bbox']

        # Rectangle rouge pour les déchets
        draw.rectangle([x1, y1, x2, y2], outline="red", width=15)

        # Texte avec classification si disponible
        if i < len(classifications) and classifications[i][0]:
            waste_type = classifications[i][0]
            confidence = classifications[i][1]
            label = f"{waste_type} ({confidence:.0%})"
        else:
            label = f"Déchet ({obj['confidence']:.0%})"

        # Fond pour le texte
        text_bbox = draw.textbbox((x1, y1-30), label, font=font)
        draw.rectangle(text_bbox, fill="red")
        draw.text((x1, y1-30), label, fill="white", font=font)

    # Annoter les non-déchets (vert)
    for obj in non_waste_objects:
        x1, y1, x2, y2 = obj['bbox']

        # Rectangle vert pour les non-déchets
        draw.rectangle([x1, y1, x2, y2], outline="green", width=15)

        label = f"{obj['class']} ({obj['confidence']:.0%})"

        # Fond pour le texte
        text_bbox = draw.textbbox((x1, y1-30), label, font=font)
        draw.rectangle(text_bbox, fill="green")
        draw.text((x1, y1-30), label, fill="white", font=font)

    return annotated_image

def display_detection_results(waste_objects, non_waste_objects, classifications):
    """Affichage détaillé des résultats de détection"""

    st.markdown("### 📊 Résultats de l'analyse")

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("#### 🗑 Déchets détectés")
        if waste_objects:
            for i, obj in enumerate(waste_objects):
                with st.expander(f"Déchet #{i+1} - Confiance: {obj['confidence']:.0%}"):
                    if i < len(classifications) and classifications[i][0]:
                        st.markdown(f"*Type:* {classifications[i][0]}")
                        st.markdown(f"*Confiance classification:* {classifications[i][1]:.0%}")
                    else:
                        st.markdown("*Type:* Non classifié")

                    x1, y1, x2, y2 = obj['bbox']
                    st.markdown(f"*Position:* ({x1}, {y1}) → ({x2}, {y2})")
        else:
            st.info("Aucun déchet détecté dans cette image.")

    with col2:
        st.markdown("#### ✅ Objets non-déchets")
        if non_waste_objects:
            for i, obj in enumerate(non_waste_objects):
                with st.expander(f"Objet #{i+1} - {obj['class']}"):
                    st.markdown(f"*Confiance:* {obj['confidence']:.0%}")
                    x1, y1, x2, y2 = obj['bbox']
                    st.markdown(f"*Position:* ({x1}, {y1}) → ({x2}, {y2})")
        else:
            st.info("Aucun objet non-déchet détecté.")
# Interface principale modifiée
def main():
    # Sidebar améliorée avec onglets
    with st.sidebar:
        st.markdown("""
        <div class="sidebar-info">
            <h2>🤖 Smart AI Detection</h2>
            <p>ENSAM Meknès - 2025</p>
        </div>
        """, unsafe_allow_html=True)

        tab1, tab2 = st.tabs(["🔧 Paramètres", "💬 Chatbot"])

        with tab1:
            confidence_threshold = st.slider("Seuil de confiance", 0.1, 1.0, 0.25, 0.05)
            show_details = st.checkbox("Afficher les détails", value=True)

        with tab2:
            st.markdown("### Assistant Virtuel")
            st.markdown("Posez des questions sur:")
            st.markdown("- ♻ Recyclage")
            st.markdown("- 🗑 Localisation des poubelles")
            st.markdown("- 🏢 Points de collecte")

    # Header principal
    st.markdown("""
    <div class="main-header">
        <h1>🚀 Smart Waste Detection</h1>
        <p>Intelligence Artificielle pour la détection et classification des déchets</p>
    </div>
    """, unsafe_allow_html=True)

# Onglets principaux - MODIFIÉ
    tab_detection, tab_chatbot, tab_statistics = st.tabs(["🔍 Détection d'Images", "💬 Assistant Virtuel", "📊 Statistiques"])

    with tab_detection:
        # Chargement des modèles
        model_detect, model_classify = load_models()

        if model_detect is None or model_classify is None:
            st.error("Impossible de charger les modèles. Vérifiez les chemins des fichiers.")
            st.stop()

        st.success("✅ Modèles chargés avec succès !")

        # Interface améliorée avec sélection de lieu - NOUVEAU
        selected_location, uploaded_files = enhanced_detection_interface()

        if uploaded_files:
            # Initialiser la base de données
            init_database()

            # Statistiques globales
            total_images = len(uploaded_files)
            total_detected = 0
            total_waste = 0
            total_non_waste = 0
            all_waste_types = []

            # Barre de progression
            progress_bar = st.progress(0)
            status_text = st.empty()

            # Conteneur pour les résultats
            results_container = st.container()

            # Traitement de chaque image
            for idx, uploaded_file in enumerate(uploaded_files):
                # Mise à jour de la progression
                progress = (idx + 1) / total_images
                progress_bar.progress(progress)
                status_text.text(f"Traitement de l'image {idx + 1}/{total_images}: {uploaded_file.name}")

                # Chargement de l'image
                image = Image.open(uploaded_file).convert("RGB")

                with results_container:
                    st.markdown(f'<div class="analysis-header">📸 Analyse: {uploaded_file.name}</div>', unsafe_allow_html=True)

                    # Affichage image originale
                    col1, col2 = st.columns(2)

                    with col1:
                        st.markdown("#### Image originale")
                        st.markdown('<div class="image-container">', unsafe_allow_html=True)
                        st.image(image, use_column_width=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    # Traitement IA
                    with st.spinner("🤖 Analyse IA en cours..."):
                        # 1. Détection des objets (déchets vs non-déchets)
                        waste_objects, non_waste_objects, detection_result = process_image_detection(image, model_detect)

                        # 2. Classification des déchets détectés
                        classifications = []
                        for waste_obj in waste_objects:
                            waste_type, confidence = classify_waste(image, waste_obj['bbox'], model_classify)
                            classifications.append((waste_type, confidence))
                            if waste_type:
                                all_waste_types.append(waste_type)

                        # 3. Création de l'image annotée
                        annotated_image = create_annotated_image(image, waste_objects, non_waste_objects, classifications)

                    with col2:
                        st.markdown("#### Image analysée")
                        st.markdown('<div class="image-container">', unsafe_allow_html=True)
                        st.image(annotated_image, use_column_width=True)
                        st.markdown('</div>', unsafe_allow_html=True)

                    # Statistiques pour cette image
                    st.markdown('<div class="detection-card">', unsafe_allow_html=True)
                    cols = st.columns(4)

                    with cols[0]:
                        st.markdown(f"""
                        <div class="stat-card">
                            <div class="stat-number">{len(waste_objects) + len(non_waste_objects)}</div>
                            <div class="stat-label">Objets détectés</div>
                        </div>
                        """, unsafe_allow_html=True)

                    with cols[1]:
                        st.markdown(f"""
                        <div class="stat-card" style="background: linear-gradient(135deg, #ff6b6b 0%, #ee5a24 100%);">
                            <div class="stat-number">{len(waste_objects)}</div>
                            <div class="stat-label">Déchets</div>
                        </div>
                        """, unsafe_allow_html=True)

                    with cols[2]:
                        st.markdown(f"""
                        <div class="stat-card" style="background: linear-gradient(135deg, #00d2d3 0%, #54a0ff 100%);">
                            <div class="stat-number">{len(non_waste_objects)}</div>
                            <div class="stat-label">Non-déchets</div>
                        </div>
                        """, unsafe_allow_html=True)

                    with cols[3]:
                        classified_waste = sum(1 for c in classifications if c[0] is not None)
                        st.markdown(f"""
                        <div class="stat-card" style="background: linear-gradient(135deg, #ffa726 0%, #fb8c00 100%);">
                            <div class="stat-number">{classified_waste}</div>
                            <div class="stat-label">Classifiés</div>
                        </div>
                        """, unsafe_allow_html=True)

                    st.markdown('</div>', unsafe_allow_html=True)

                    # NOUVEAU - Interface de sauvegarde
                    show_save_confirmation(waste_objects, classifications, selected_location, uploaded_file.name)

                    # Détails des détections
                    if show_details:
                        display_detection_results(waste_objects, non_waste_objects, classifications)

                    # Bouton de téléchargement
                    buf = BytesIO()
                    annotated_image.save(buf, format="PNG")
                    st.download_button(
                        label="📥 Télécharger l'image annotée",
                        data=buf.getvalue(),
                        file_name=f"analysed_{uploaded_file.name}",
                        mime="image/png"
                    )

                    # Mise à jour des totaux
                    total_detected += len(waste_objects) + len(non_waste_objects)
                    total_waste += len(waste_objects)
                    total_non_waste += len(non_waste_objects)

                    st.markdown("---")

            # [Reste de votre code existant pour les graphiques globaux...]

        else:
            # [Votre message d'accueil existant...]
            pass

    with tab_chatbot:
        chatbot = load_chatbot()
        st.markdown("### 💬 Assistant de Gestion des Déchets")
        st.markdown("Posez vos questions sur le recyclage et la localisation des poubelles")

        # Initialisation de l'historique
        if 'chat_history' not in st.session_state:
            st.session_state.chat_history = []

        # Affichage de l'historique
        for msg in st.session_state.chat_history:
            with st.chat_message(msg["role"]):
                st.markdown(msg["content"])

        # Entrée utilisateur
        if prompt := st.chat_input("Votre question..."):
            # Ajout du message utilisateur
            with st.chat_message("user"):
                st.markdown(prompt)

            st.session_state.chat_history.append({
                'role': 'user',
                'content': prompt
            })

            # Réponse du chatbot
            with st.spinner("Recherche en cours..."):
                response = chatbot.process_query(prompt)

            with st.chat_message("assistant"):
                st.markdown(response)
            st.session_state.chat_history.append({
                'role': 'assistant',
                'content': response
            })

        pass

    # NOUVEAU - Onglet Statistiques
    with tab_statistics:
        add_statistics_css()
        display_statistics_dashboard()

if __name__ == "__main__":
    main()

Overwriting app.py


# **🌐 Étape 3 – Obtenir ton IP publique (optionnel)**

In [ ]:
!wget -q -O - ipv4.icanhazip.com


34.53.6.180


# **🚀 Étape 4 – Lancer l'application et ouvrir un tunnel**

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501




⠙⠹⠸⠼
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.196.179.32:8501

⠴⠦⠧⠇⠏⠋your url is: https://blue-suns-pay.loca.lt
